# Reading a trial registry record against its paper

**Data kit lab** `actt1-registry-vs-paper` · Clinical · about 30 minutes

**The question.** Do the results posted on ClinicalTrials.gov for the ACTT-1 trial of remdesivir (NCT04280705) carry the same primary outcome and the same participant counts as the trial's final report in the New England Journal of Medicine?

**What you do.** You open one registry record through the public API and walk through it.

1. Print the date the registry data were processed.
2. Open the record and see its sections.
3. Find the arms and the groups of each results table.
4. Read participant flow and baseline.
5. Read the primary outcome and the analysis attached to it.
6. Read the adverse events table.
7. Compare your numbers with the paper.

**The data.** Source: ClinicalTrials.gov, a database of the U.S. National Library of Medicine, used under its Terms and Conditions. Nothing is copied into this notebook: the first cell downloads the record from the source (about 0.2 MB). The record holds summary tables only. There is no patient-level data in it.

**The paper.** Beigel JH and others. Remdesivir for the Treatment of Covid-19 - Final Report. N Engl J Med 2020. https://doi.org/10.1056/NEJMoa2007764 . Keep it open at Table 2 while you work.

**You need.** Python 3.10 or later with `pandas`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'actt1-registry-vs-paper'
FILES = json.loads(r'''{
 "study": {
  "url": "https://clinicaltrials.gov/api/v2/studies/NCT04280705?fields=protocolSection,resultsSection,documentSection,hasResults",
  "name": "study",
  "sha256": "9eec5f0cd485357adfd8159e560c414e6d10b9d63330420afd7b652aa229f6c5",
  "bytes": 212070
 },
 "version": {
  "url": "https://clinicaltrials.gov/api/v2/version",
  "name": "version",
  "sha256": "dea044878c7aea78177cd07e98ed31b379112d63ee5391c0d04810970baff87c",
  "bytes": 71
 }
}''')
PUBLISHED = json.loads(r'''{
 "n_itt_remdesivir": {
  "label": "Randomized (N), remdesivir",
  "value": 541,
  "tolerance": {
   "exact": true
  }
 },
 "n_itt_placebo": {
  "label": "Randomized (N), placebo",
  "value": 521,
  "tolerance": {
   "exact": true
  }
 },
 "median_recovery_remdesivir": {
  "label": "Median days to recovery, remdesivir",
  "value": 10.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "median_ci_low_remdesivir": {
  "label": "Median 95% CI lower, remdesivir",
  "value": 9.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "median_ci_high_remdesivir": {
  "label": "Median 95% CI upper, remdesivir",
  "value": 11.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "median_recovery_placebo": {
  "label": "Median days to recovery, placebo",
  "value": 15.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "median_ci_low_placebo": {
  "label": "Median 95% CI lower, placebo",
  "value": 13.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "median_ci_high_placebo": {
  "label": "Median 95% CI upper, placebo",
  "value": 18.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "rate_ratio": {
  "label": "Rate ratio for recovery",
  "value": 1.29,
  "tolerance": {
   "abs": 0.005
  }
 },
 "rate_ratio_ci_low": {
  "label": "Rate ratio 95% CI lower",
  "value": 1.12,
  "tolerance": {
   "abs": 0.005
  }
 },
 "rate_ratio_ci_high": {
  "label": "Rate ratio 95% CI upper",
  "value": 1.49,
  "tolerance": {
   "abs": 0.005
  }
 },
 "p_value": {
  "label": "P value, time to recovery",
  "value": "<0.001",
  "tolerance": {
   "exact": true
  }
 },
 "deaths_day29_remdesivir": {
  "label": "Deaths by day 29, remdesivir",
  "value": 59,
  "tolerance": {
   "exact": true
  }
 },
 "deaths_day29_placebo": {
  "label": "Deaths by day 29, placebo",
  "value": 77,
  "tolerance": {
   "exact": true
  }
 },
 "n_male_remdesivir": {
  "label": "Male participants, remdesivir",
  "value": 352,
  "tolerance": {
   "exact": true
  }
 },
 "n_male_placebo": {
  "label": "Male participants, placebo",
  "value": 332,
  "tolerance": {
   "exact": true
  }
 },
 "n_randomized_total": {
  "label": "Randomized, both arms",
  "value": 1062,
  "tolerance": {
   "exact": true
  }
 },
 "n_treated_remdesivir": {
  "label": "Received treatment as assigned, remdesivir",
  "value": 531,
  "tolerance": {
   "exact": true
  }
 },
 "n_treated_placebo": {
  "label": "Received treatment as assigned, placebo",
  "value": 517,
  "tolerance": {
   "exact": true
  }
 },
 "n_completed_remdesivir": {
  "label": "Completed, recovered or died, remdesivir",
  "value": 517,
  "tolerance": {
   "exact": true
  }
 },
 "n_completed_placebo": {
  "label": "Completed, recovered or died, placebo",
  "value": 508,
  "tolerance": {
   "exact": true
  }
 },
 "n_as_treated_remdesivir": {
  "label": "As-treated population, remdesivir",
  "value": 532,
  "tolerance": {
   "exact": true
  }
 },
 "n_as_treated_placebo": {
  "label": "As-treated population, placebo",
  "value": 516,
  "tolerance": {
   "exact": true
  }
 },
 "n_sae_remdesivir": {
  "label": "With a serious adverse event, remdesivir",
  "value": 131,
  "tolerance": {
   "exact": true
  }
 },
 "n_sae_placebo": {
  "label": "With a serious adverse event, placebo",
  "value": 163,
  "tolerance": {
   "exact": true
  }
 }
}''')
PUBLISHED_FROM = 'Europe PMC full-text record PMC7262788 (XML, no page numbers): Table 2, Outcomes Overall and According to Score on the Ordinal Scale in the Intention-to-Treat Population, Overall columns; Table 1; Results text, sections Patients and Safety Outcomes | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC7262788/fullTextXML'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. When were the registry data processed?

The registry's terms ask anyone who shows its data to display the date the data were processed. The API has a small file for that.

The first cell above checks each download against the copy this lab was tested with. The `version` file changes every time the registry refreshes its data, so a notice that it differs is expected. The same notice for the `study` file means the record itself changed (see section 2).

In [ ]:
import json

import pandas as pd

version = json.loads(data_file("version").read_text())
print("API version:", version["apiVersion"])
print("Data processed by ClinicalTrials.gov:", version["dataTimestamp"])

## 2. Open the record

A registry record is one JSON document per trial. ClinicalTrials.gov says its data are provided by the organisations that sponsor and run the studies. The party answerable for a record is called the responsible party.

The record has a protocol section (what was planned) and, once results are posted, a results section (what was found). Each section is split into modules.

In [ ]:
study = json.loads(data_file("study").read_text())
print("Top-level sections:", list(study))
protocol, results = study["protocolSection"], study["resultsSection"]
print("Protocol modules:", len(protocol))
for name in protocol:
    print("   ", name)
print("Results modules:", len(results))
for name in results:
    print("   ", name)

Now the facts that identify the record. Note two dates: when results were first posted and when the record was last updated. A record can be edited after posting, so write both dates next to any number you take from it.

The record also lists its documents (protocol, analysis plan, consent form) and the papers linked to it.

In [ ]:
ident = protocol["identificationModule"]
status = protocol["statusModule"]
sponsor = protocol["sponsorCollaboratorsModule"]
print(ident["nctId"], "|", ident["briefTitle"])
print("Sponsor:", sponsor["leadSponsor"]["name"], "| responsible party:", sponsor["responsibleParty"]["type"])
print("Results first posted:", status["resultsFirstPostDateStruct"]["date"])
print("Record last updated: ", status["lastUpdatePostDateStruct"]["date"])
print("Enrolment:", protocol["designModule"]["enrollmentInfo"])
for doc in study["documentSection"]["largeDocumentModule"]["largeDocs"]:
    print("Document:", doc["label"], "|", doc["filename"], "| dated", doc["date"])
for ref in protocol["referencesModule"]["references"]:
    if "Final Report" in ref["citation"]:
        print("Cited paper: PMID", ref["pmid"], "|", ref["citation"][-93:])

## 3. Arms and groups

The protocol section names the arms. Each results module then defines its own groups, with its own ids.

**What goes wrong.** The ids are different in every module, and the order is the registry's, not the paper's. Here Placebo comes first; Table 2 of the paper prints Remdesivir first. Baseline has a third group, Total.

In [ ]:
for arm in protocol["armsInterventionsModule"]["armGroups"]:
    print("Arm:", arm["label"], "|", arm["type"])
print()
measures = results["outcomeMeasuresModule"]["outcomeMeasures"]
group_lists = {
    "participant flow": results["participantFlowModule"]["groups"],
    "baseline": results["baselineCharacteristicsModule"]["groups"],
    "first outcome": measures[0]["groups"],
    "adverse events": results["adverseEventsModule"]["eventGroups"],
}
for name, groups in group_lists.items():
    print(f"{name:<17}", [(g["id"], g["title"]) for g in groups])

So we never pick a group by position. This helper finds a group by its title and stops if the title is missing or repeated.

In [ ]:
ARMS = {"remdesivir": "Remdesivir", "placebo": "Placebo"}


def group_id(groups, title):
    """The id of the one group with this title."""
    ids = [g["id"] for g in groups if g["title"] == title]
    assert len(ids) == 1, f"{len(ids)} groups are titled {title!r}"
    return ids[0]


for key, title in ARMS.items():
    print(title, "is", group_id(measures[0]["groups"], title), "in the first outcome measure")

## 4. Participant flow

Participant flow is the registry's version of the flow diagram in a paper: how many people started, reached each milestone and finished. The counts are stored as text, so we convert them.

In [ ]:
flow = results["participantFlowModule"]
print(flow["recruitmentDetails"])
print()
rows = []
for period in flow["periods"]:
    for milestone in period["milestones"]:
        for cell in milestone["achievements"]:
            rows.append({"milestone": milestone["type"], "group": cell["groupId"], "n": int(cell["numSubjects"])})
long = pd.DataFrame(rows)
flow_table = long.pivot(index="milestone", columns="group", values="n").reindex(long["milestone"].unique())
flow_table = flow_table.rename(columns={g["id"]: g["title"] for g in flow["groups"]})
print(flow_table.to_string())
print()
for reason in flow["periods"][0]["dropWithdraws"]:
    print(f"{reason['type']:<36}", {cell["groupId"]: cell["numSubjects"] for cell in reason["reasons"]})

## 5. Baseline

The baseline module holds the registry's version of a paper's Table 1. Each measure has classes and categories, and one value per group. We read the sex counts.

In [ ]:
base = results["baselineCharacteristicsModule"]
base_title = {g["id"]: g["title"] for g in base["groups"]}
print("Baseline denominators:", {base_title[c["groupId"]]: int(c["value"]) for c in base["denoms"][0]["counts"]})
print("Baseline measures:", [m["title"] for m in base["measures"]])
sex = next(m for m in base["measures"] if m["title"] == "Sex: Female, Male")
male = next(cat for cat in sex["classes"][0]["categories"] if cat["title"] == "Male")
n_male = {base_title[cell["groupId"]]: int(cell["value"]) for cell in male["measurements"]}
print("Male participants:", n_male)

## 6. Outcome measures

This is the core of the results section. One outcome measure holds:

- its type (primary or secondary), title, description, time frame and unit;
- the kind of number posted (`paramType`, for example a median) and its spread (`dispersionType`);
- its own groups and the number of participants analyzed in each (`denoms`);
- the values, nested as classes, then categories, then one measurement per group;
- zero or more statistical analyses that compare the groups.

Nested data is hard to scan, so we flatten it to one row per measure, class, category and arm.

In [ ]:
print("Outcome measures posted:", len(measures))
rows = []
for number, om in enumerate(measures):
    title_of = {g["id"]: g["title"] for g in om["groups"]}
    for cls in om["classes"]:
        for cat in cls["categories"]:
            for cell in cat["measurements"]:
                rows.append({
                    "measure": number, "type": om["type"], "title": om["title"],
                    "class": cls.get("title", ""), "category": cat.get("title", ""),
                    "param": om["paramType"], "unit": om["unitOfMeasure"], "arm": title_of[cell["groupId"]],
                    "value": cell.get("value"), "lower": cell.get("lowerLimit"), "upper": cell.get("upperLimit"),
                })
flat = pd.DataFrame(rows)
print("Rows after flattening:", len(flat))
print()
print(flat.drop_duplicates("measure")["type"].value_counts().to_string())
print()
print("Measures of type PRIMARY:")
print(flat.loc[flat["type"] == "PRIMARY", "title"].drop_duplicates().to_string(index=False))

**What goes wrong.** More than one measure is marked PRIMARY, because the sponsor posted the primary outcome again for subgroups. Filtering on the type alone is not enough. We ask for the type and the exact title, and check that one measure comes back.

In [ ]:
found = [om for om in measures if om["type"] == "PRIMARY" and om["title"] == "Time to Recovery"]
assert len(found) == 1, f"{len(found)} measures match"
primary = found[0]
print(primary["title"], "|", primary["paramType"], "|", primary["dispersionType"], "|", primary["unitOfMeasure"], "|", primary["timeFrame"])
print("Population: ", primary["populationDescription"])
print("Description:", primary["description"])
print()
cells = primary["classes"][0]["categories"][0]["measurements"]
print("Values as stored:", cells)

The values are text (`"15"`, not `15`). We convert them, and we pick each arm by its title.

In [ ]:
n_analyzed = {c["groupId"]: int(c["value"]) for c in primary["denoms"][0]["counts"]}
outcome = {}
for key, title in ARMS.items():
    gid = group_id(primary["groups"], title)
    cell = next(c for c in cells if c["groupId"] == gid)
    outcome[key] = {"n": n_analyzed[gid], "median": float(cell["value"]),
                    "low": float(cell["lowerLimit"]), "high": float(cell["upperLimit"])}
    print(f"{title:<11}", outcome[key])

## 7. The analysis attached to the primary outcome

The medians describe each arm. The comparison between the arms is stored separately, in `analyses`. An analysis names the groups it compares, the test, the p-value and the estimate with its confidence interval.

**What goes wrong.** The names differ from the paper. The registry labels this estimate `Cox Proportional Hazard` and the method `Log Rank`. The paper calls the same number the rate ratio for recovery. The footnote of its Table 2 says it comes from a stratified Cox model, with the p-value from a stratified log-rank test. The event here is recovery, so a ratio above 1 favours remdesivir.

The p-value is `"<0.001"`. It cannot be turned into a number, so we keep it as text.

In [ ]:
print("Analyses attached:", len(primary["analyses"]))
analysis = primary["analyses"][0]
for field, value in analysis.items():
    print(f"   {field:<20}{value}")
print("Groups compared:", [g["title"] for g in primary["groups"] if g["id"] in analysis["groupIds"]])
ratio = {"value": float(analysis["paramValue"]), "low": float(analysis["ciLowerLimit"]),
         "high": float(analysis["ciUpperLimit"]), "p": analysis["pValue"]}
print(ratio)

## 8. Adverse events

The adverse events module has three tables: deaths from any cause, serious adverse events, and other adverse events. Each group carries a number affected and a number at risk. Here the counts are stored as numbers, not text.

**What goes wrong.** The number at risk is not the same in every table. Read the module's own description, printed first below, before you compute a percentage.

In [ ]:
events = results["adverseEventsModule"]
print(events["description"])
print()
safety = {}
for key, title in ARMS.items():
    safety[key] = next(g for g in events["eventGroups"] if g["title"] == title)
    g = safety[key]
    print(f"{title:<11} deaths {g['deathsNumAffected']} of {g['deathsNumAtRisk']} | serious adverse events {g['seriousNumAffected']} of {g['seriousNumAtRisk']}")

## 9. One quantity, posted twice

Serious adverse events appear in two places in this record: as counts in the adverse events module (section 8), and as a percentage in one of the outcome measures. Compute the percentage from the counts and put it next to the posted one.

In [ ]:
posted = flat.loc[flat["title"] == "Percentage of Participants Reporting Serious Adverse Events (SAEs)", ["arm", "value", "lower", "upper"]]
print("Posted as an outcome measure (percent, with 95% limits):")
print(posted.to_string(index=False))
print()
print("Computed from the adverse events module:")
for key, title in ARMS.items():
    g = safety[key]
    print(f"   {title:<11}{g['seriousNumAffected']} / {g['seriousNumAtRisk']} = {100 * g['seriousNumAffected'] / g['seriousNumAtRisk']:.1f} percent")

If the two do not agree for an arm, trust the counts: a count cannot be rounded wrongly. The paper prints its own figure for this in the Results text, under Safety Outcomes. Look it up and see which of the two it matches.

## 10. Check against the paper

Each number you read maps to one place in the paper.

| Registry field | Where the paper prints it |
|---|---|
| Primary outcome, participants analyzed (`denoms`) | Table 2, header of the two Overall columns |
| Primary outcome, value and limits per arm | Table 2, row "Median time to recovery (95% CI)" |
| Primary outcome analysis, estimate, limits, p-value | Table 2, row "Rate ratio (95% CI)" |
| Adverse events, deaths | Table 2, row "No. of deaths by day 29" |
| Baseline, sex | Table 1, row "Male sex" |
| Enrolment count; participant flow milestones | Results text, section Patients |
| Adverse events, serious events and number at risk | Results text, sections Patients and Safety Outcomes |

`PUBLISHED` holds the paper's values. They were copied by code from the full text of the paper, not typed. `report` prints your value beside each one. Counts and the p-value must match exactly; the medians and the ratio must be within half a unit of the last digit the paper prints.

In [ ]:
flow_n = {key: flow_table[title] for key, title in ARMS.items()}
RESULTS = {
    "rate_ratio": ratio["value"],
    "rate_ratio_ci_low": ratio["low"],
    "rate_ratio_ci_high": ratio["high"],
    "p_value": ratio["p"],
    "n_randomized_total": int(protocol["designModule"]["enrollmentInfo"]["count"]),
}
for key, title in ARMS.items():
    RESULTS[f"n_itt_{key}"] = outcome[key]["n"]
    RESULTS[f"median_recovery_{key}"] = outcome[key]["median"]
    RESULTS[f"median_ci_low_{key}"] = outcome[key]["low"]
    RESULTS[f"median_ci_high_{key}"] = outcome[key]["high"]
    RESULTS[f"deaths_day29_{key}"] = int(safety[key]["deathsNumAffected"])
    RESULTS[f"n_male_{key}"] = n_male[title]
    RESULTS[f"n_treated_{key}"] = int(flow_n[key]["Received Treatment"])
    RESULTS[f"n_completed_{key}"] = int(flow_n[key]["COMPLETED"])
    RESULTS[f"n_as_treated_{key}"] = int(safety[key]["seriousNumAtRisk"])
    RESULTS[f"n_sae_{key}"] = int(safety[key]["seriousNumAffected"])
report(RESULTS)

## What the result says

Read the rate ratio and its confidence interval. Does the interval include 1? Then read the first paragraph of the paper's Discussion. Does it say the same?

Note what you did not find in the record: the number of recoveries, and the columns of Table 2 by baseline ordinal score. A registry record and a paper overlap; neither contains the other.

## Try next

1. The record has two mortality measures ("14-day Participant Mortality" and "29-day Participant Mortality"). Print them from `flat`. Find the mortality rows in Table 2 of the paper. How many digits does each source give, and do they agree once you round?
2. The record posts time to recovery by sex, race and ethnicity. Read the analysis for each class. Does the paper report the same subgroups?
3. Look up `dispersionType` and `paramType` in the field list at https://clinicaltrials.gov/api/v2/studies/metadata . What other values can they take?
4. Remove the title from the filter in section 6 and keep only the type. What does the `assert` tell you?
5. Open the record in a browser at https://clinicaltrials.gov/study/NCT04280705 and find the posted results. Match one table on the screen to the JSON you read.